# convT-kernel-axis-swap — worked example 1: Swap a ConvTranspose2d kernel into Conv2d layout with einops

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `convT-kernel-axis-swap`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A `ConvTranspose2d` weight is stored as `(IC, OC, KH, KW)` — in-channels first — which is the *opposite* of a `Conv2d` weight `(OC, IC, KH, KW)`. To reinterpret a transposed-conv kernel in conv layout you swap the first two axes (channels) while leaving the spatial axes alone. In einops this is the pattern `'i o kh kw -> o i kh kw'`, equivalent to `.transpose(0, 1)`.

## Worked solution

**Goal.** Take a kernel in transposed-conv layout `(IC, OC, KH, KW)` and produce the same kernel in conv layout `(OC, IC, KH, KW)`.

**Step 1 — identify the axes.** Name the four axes explicitly: `i` = in-channels, `o` = out-channels, `kh`/`kw` = kernel height/width. The transposed-conv tensor orders them `i o kh kw`.

**Step 2 — write the rearrange.** Conv layout wants out-channels first, so the target ordering is `o i kh kw`. The full pattern is `'i o kh kw -> o i kh kw'`. einops sees that only the first two named axes change position, so it issues a single transpose under the hood — no data is copied or reshaped, the spatial axes are untouched.

**Step 3 — why only the channel axes move.** This is *not* the kernel flip. The flip (reversing the spatial axes) is a separate operation needed only when you want the full-convolution equivalence; the bare layout conversion between the two module conventions is purely a channel-axis swap. Reversing `kh`/`kw` here would silently corrupt the kernel.

**Step 4 — verify.** The result has shape `(OC, IC, KH, KW)` and is bit-for-bit identical to `w.transpose(0, 1)`, confirming einops did exactly the axis swap and nothing else.

In [ ]:
def convT_to_conv_swap(w_convT: Tensor) -> Tensor:
    # (IC, OC, KH, KW)  ->  (OC, IC, KH, KW): swap the two channel axes only
    return rearrange(w_convT, 'i o kh kw -> o i kh kw')

t.manual_seed(0)
w_convT = t.randn(3, 5, 4, 4)        # ConvTranspose2d-style: (IC=3, OC=5, KH=4, KW=4)
w_conv = convT_to_conv_swap(w_convT)
print('convT weight shape (IC,OC,KH,KW):', tuple(w_convT.shape))
print('conv  weight shape (OC,IC,KH,KW):', tuple(w_conv.shape))
print('matches .transpose(0,1):', t.equal(w_conv, w_convT.transpose(0, 1)))